# 02. Attention Model Comparison

**Objective:** Compare performance of all trained models with different attention mechanisms.

**Prerequisites:**
- All 7 models must be trained
- Run `scripts/evaluate_models.py` first

---

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Evaluation Results

In [ ]:
# Load metrics
metrics_file = Path('../results/metrics/all_metrics.json')

if not metrics_file.exists():
    print(f"❌ Metrics file not found: {metrics_file}")
    print("\nPlease run evaluation first:")
    print("  uv run python scripts/evaluate_models.py")
else:
    with open(metrics_file) as f:
        metrics = json.load(f)
    
    df = pd.DataFrame(metrics)
    print(f"✓ Loaded metrics for {len(df)} models")
    print(f"\nColumns: {list(df.columns)}")

## 2. Performance Comparison Table

In [ ]:
if 'df' in locals():
    # Select key metrics
    display_cols = ['model_name', 'accuracy', 'tb_sensitivity', 'tb_precision', 
                    'tb_f1', 'f1_macro', 'params_millions']
    
    display_df = df[display_cols].copy()
    display_df = display_df.sort_values('tb_sensitivity', ascending=False)
    
    # Format for display
    formatted_df = display_df.copy()
    for col in ['accuracy', 'tb_sensitivity', 'tb_precision', 'tb_f1', 'f1_macro']:
        formatted_df[col] = formatted_df[col].apply(lambda x: f"{x:.4f}")
    formatted_df['params_millions'] = formatted_df['params_millions'].apply(lambda x: f"{x:.1f}M")
    
    print("\n📊 Model Performance Comparison (Sorted by TB Sensitivity)")
    print("=" * 80)
    print(formatted_df.to_string(index=False))
    print("=" * 80)

## 3. TB Sensitivity Comparison

In [ ]:
if 'df' in locals():
    # Extract backbone and attention type
    df['backbone'] = df['model_name'].apply(lambda x: 'resnet50' if 'resnet' in x else 'densenet121')
    df['attention'] = df['model_name'].apply(
        lambda x: 'none' if 'baseline' in x else 
                 'cbam' if 'cbam' in x else 
                 'se' if 'se' in x else 
                 'eca' if 'eca' in x else 'none'
    )
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Grouped bar chart
    pivot_resnet = df[df['backbone'] == 'resnet50'].pivot(
        index='attention', columns='backbone', values='tb_sensitivity'
    ).reset_index()
    
    # Reorder
    order = ['none', 'se', 'eca', 'cbam']
    pivot_resnet['attention'] = pd.Categorical(
        pivot_resnet['attention'], categories=order, ordered=True
    )
    pivot_resnet = pivot_resnet.sort_values('attention')
    
    # ResNet50 comparison
    sns.barplot(data=df[df['backbone'] == 'resnet50'], 
                x='attention', y='tb_sensitivity', 
                ax=axes[0], palette='Reds_r')
    axes[0].set_title('ResNet50: Attention Comparison', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Attention Type')
    axes[0].set_ylabel('TB Sensitivity')
    axes[0].set_ylim(0, 1)
    
    # Add value labels
    for i, row in df[df['backbone'] == 'resnet50'].sort_values('attention').iterrows():
        axes[0].text(i, row['tb_sensitivity'] + 0.02, f"{row['tb_sensitivity']:.3f}", 
                    ha='center', fontsize=10)
    
    # DenseNet121 comparison
    densenet_df = df[df['backbone'] == 'densenet121']
    if not densenet_df.empty:
        sns.barplot(data=densenet_df, x='attention', y='tb_sensitivity', 
                    ax=axes[1], palette='Blues_r')
        axes[1].set_title('DenseNet121: Attention Comparison', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Attention Type')
        axes[1].set_ylabel('TB Sensitivity')
        axes[1].set_ylim(0, 1)
        
        for i, row in densenet_df.sort_values('attention').iterrows():
            axes[1].text(i, row['tb_sensitivity'] + 0.02, f"{row['tb_sensitivity']:.3f}", 
                        ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('../results/figures/05_attention_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: ../results/figures/05_attention_comparison.png")

## 4. Accuracy vs TB Sensitivity Trade-off

In [ ]:
if 'df' in locals():
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Scatter plot
    for idx, row in df.iterrows():
        if 'cbam' in row['model_name']:
            color, marker, s = '#e74c3c', 'o', 200
        elif 'se' in row['model_name']:
            color, marker, s = '#3498db', 's', 150
        elif 'eca' in row['model_name']:
            color, marker, s = '#2ecc71', '^', 150
        else:
            color, marker, s = '#95a5a6', 'D', 150
        
        ax.scatter(row['accuracy'], row['tb_sensitivity'], s=s, c=color, 
                  marker=marker, alpha=0.7, label=row['model_name'])
        ax.annotate(row['model_name'], (row['accuracy'], row['tb_sensitivity']),
                   xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    ax.set_xlabel('Overall Accuracy', fontsize=12)
    ax.set_ylabel('TB Sensitivity', fontsize=12)
    ax.set_title('Accuracy vs TB Sensitivity', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/figures/06_accuracy_vs_tb_sensitivity.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: ../results/figures/06_accuracy_vs_tb_sensitivity.png")

## 5. Parameter Efficiency

In [ ]:
if 'df' in locals():
    # Calculate efficiency metric
    df['tb_sensitivity_per_param'] = df['tb_sensitivity'] / df['params_millions']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Params vs TB Sensitivity
    sns.scatterplot(data=df, x='params_millions', y='tb_sensitivity', 
                   hue='attention', size='tb_sensitivity',
                   sizes=(100, 300), ax=axes[0], palette='viridis')
    axes[0].set_title('Parameters vs TB Sensitivity', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Parameters (Millions)')
    axes[0].set_ylabel('TB Sensitivity')
    
    # Efficiency comparison
    sns.barplot(data=df.sort_values('tb_sensitivity_per_param', ascending=False),
               x='model_name', y='tb_sensitivity_per_param',
               ax=axes[1], palette='viridis')
    axes[1].set_title('TB Sensitivity per Parameter (Efficiency)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('TB Sensitivity / Million Params')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../results/figures/07_parameter_efficiency.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: ../results/figures/07_parameter_efficiency.png")

## 6. Summary Statistics

In [ ]:
if 'df' in locals():
    print("\n" + "=" * 80)
    print("📊 ATTENTION COMPARISON SUMMARY")
    print("=" * 80)
    
    # Best TB sensitivity
    best_tb = df.loc[df['tb_sensitivity'].idxmax()]
    print(f"\n🏆 Best TB Sensitivity: {best_tb['model_name']} ({best_tb['tb_sensitivity']:.4f})")
    
    # Best accuracy
    best_acc = df.loc[df['accuracy'].idxmax()]
    print(f"🏆 Best Accuracy: {best_acc['model_name']} ({best_acc['accuracy']:.4f})")
    
    # Most efficient
    best_eff = df.loc[df['tb_sensitivity_per_param'].idxmax()]
    print(f"⚡ Most Efficient: {best_eff['model_name']} ({best_eff['tb_sensitivity_per_param']:.4f} sens/param)")
    
    # Average improvement
    baseline_resnet = df[(df['backbone'] == 'resnet50') & (df['attention'] == 'none')]['tb_sensitivity'].values[0]
    cbam_resnet = df[(df['backbone'] == 'resnet50') & (df['attention'] == 'cbam')]['tb_sensitivity'].values[0]
    improvement = ((cbam_resnet - baseline_resnet) / baseline_resnet) * 100
    
    print(f"\n📈 CBAM Improvement over Baseline (ResNet50): +{improvement:.1f}%")
    
    print(f"\n📁 All figures saved to: ../results/figures/")
    print("=" * 80)